# Unintended Memorization in LLMs

Esecuzione sequenziale dei 4 script (`build_corpus.py` → `train.py` → `evaluate.py` → `analyze.py`).
Modifica eventuali parametri in `config.py`.

In [ ]:
# 0. (Optional) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Setup GPU e dipendenze
!nvidia-smi
!pip uninstall -y -q torchao # torchao preinstallato in Colab causa conflitti
!pip install -r requirements.txt

import torch, transformers, peft, datasets
print(f'torch={torch.__version__} | transformers={transformers.__version__} | peft={peft.__version__} | datasets={datasets.__version__}')


In [ ]:
# 2. Controlla i parametri in config.py
import config
print(f'Model: {config.MODEL_NAME} | Seed: {config.SEED} | LR: {config.LEARNING_RATE} | Shuffle BG: {config.SHUFFLE_BACKGROUND}')


In [ ]:
# 3. Fase 1 — Costruzione corpus 256x96 + Sanity Checks SC-1..SC-5
!python build_corpus.py


In [ ]:
# 4. Fase 2 — LoRA Fine-Tuning (2 epoche, Controlled Shuffling)
!python train.py


In [ ]:
# 5. Fase 3 — Valutazione Pre-Training e Post-Training (48 + 48 test)
!python evaluate.py


In [ ]:
# 6. Fase 4 — Statistiche descrittive, verifica H1-H5 e generazione 5 grafici
!python analyze.py


In [ ]:
# 7. Visualizzazione inline dei grafici e ispezione delle 48 generazioni post-training
import pandas as pd
from IPython.display import Image, display
import config

for img in sorted(config.PLOTS_DIR.glob('*.png')):
    print(f'--- {img.name} ---')
    display(Image(filename=str(img)))

df = pd.read_csv(config.RESULTS_DIR / 'eval_paired_48.csv')
cols = ['group', 'cumulative_exposures', 'prompt_type', 'project_id', 'secret', 'first_line_norm', 'strict_exact_match', 'error_categories', 'net_delta_ll', 'w1_rank']
df.sort_values(['prompt_type', 'cumulative_exposures', 'canary_id'])[cols]
